In [19]:
import re
from pathlib import Path
import pandas as pd
import pymupdf

In [3]:
batch_dir = "./concatenated_batches"

In [15]:
def parse_batch_pdfs(batch_dir: str, split_by_page: bool = True) -> pd.DataFrame:
    records = []
    batch_path = Path(batch_dir)

    pdf_files = sorted(list(batch_path.glob("*.pdf")))
    if not pdf_files:
        print(f"No PDFs found in {batch_dir}")
        return pd.DataFrame(columns=["file_name", "candidate_id", "parsed_resume_text"])

    for pdf_file in pdf_files:
        # Use pymupdf directly
        print(pdf_file)
        doc = pymupdf.open(pdf_file)
        print(len(doc))

        if split_by_page:
            for page_num in range(len(doc)):
                page = doc[page_num]
                text = page.get_text("text").strip()
                print(page, text[0:100])
                
                records.append({
                    "file_name": pdf_file.name,
                    "candidate_id": f"{pdf_file.stem}_p{page_num + 1}",
                    "parsed_resume_text": text
                })
        else:
            full_text = "\n".join([page.get_text("text").strip() for page in doc])
            records.append({
                "file_name": pdf_file.name,
                "candidate_id": pdf_file.stem,
                "parsed_resume_text": full_text
            })

        doc.close()
    return pd.DataFrame(records)

In [20]:
# Regex Patterns
EMAIL_REGEX = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')
# Handles common formats: (123) 456-7890, 123-456-7890, 123.456.7890, +1 123 456 7890
PHONE_REGEX = re.compile(r'(?:\+?\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}')

def normalize_phone(phone_str: str) -> str:
    """Strips non-digits to ensure (123) 456-7890 matches 123-456-7890."""
    digits = re.sub(r'\D', '', phone_str)
    return digits[-10:] if len(digits) >= 10 else digits

In [21]:
normalize_phone('(800) 123-4567')

'8001234567'

In [25]:
t1 = normalize_phone('(800) 123-4567')
t2 = normalize_phone('1-800-123-4567')
t1 == t2

True

In [48]:
def parse_batch_pdfs_regex(batch_dir: str, header_char_limit: int = 600) -> pd.DataFrame:
    records = []
    batch_path = Path(batch_dir)
    pdf_files = sorted(list(batch_path.glob("*.pdf")))

    for pdf_file in pdf_files:
        doc = pymupdf.open(pdf_file)
        
        current_candidate_text = []
        current_email = None
        current_phone = None
        candidate_count = 0

        for page_num in range(len(doc)):
            page_text = doc[page_num].get_text("text").strip()
            
            # Inspect the top portion of the page text for header details
            header_text = page_text[:header_char_limit]
            
            emails = EMAIL_REGEX.findall(header_text)
            phones = PHONE_REGEX.findall(header_text)
            
            page_email = emails[0].lower() if emails else None
            page_phone = normalize_phone(phones[0]) if phones else None

            # Decision Logic: Check if this page introduces a new candidate
            is_new_candidate = False

            if current_email is None and current_phone is None:
                # Initial candidate on Page 1
                is_new_candidate = True
            elif page_email and page_email != current_email:
                is_new_candidate = True
            elif page_phone and page_phone != current_phone:
                is_new_candidate = True

            if is_new_candidate:
                # Flush current candidate buffer to records list
                if current_candidate_text:
                    candidate_count += 1
                    records.append({
                        "file_name": pdf_file.name,
                        "candidate_id": f"{pdf_file.stem}_cand_{candidate_count}",
                        "primary_email": current_email,
                        "primary_phone": current_phone,
                        "parsed_resume_text": "\n".join(current_candidate_text)
                    })
                
                # Reset buffer for the new candidate
                current_candidate_text = [page_text]
                current_email = page_email
                current_phone = page_phone
                print('candidate_count:', candidate_count)
                print('page_email:', page_email)
                print('page_phone:', page_phone)
            else:
                # Continuation page (no new header match found)
                current_candidate_text.append(page_text)

        # Flush the final candidate in the PDF
        if current_candidate_text:
            candidate_count += 1
            records.append({
                "file_name": pdf_file.name,
                "candidate_id": f"{pdf_file.stem}_cand_{candidate_count}",
                "primary_email": current_email,
                "primary_phone": current_phone,
                "parsed_resume_text": "\n".join(current_candidate_text)
            })

        doc.close()

    df = pd.DataFrame(records)
    print(f"Processed {len(pdf_files)} PDF batches into {len(df)} candidate records.")
    return df

In [34]:
batch_path = Path(batch_dir)
pdf_files = sorted(list(batch_path.glob("*.pdf")))

In [40]:
pdf_files[0].stem

'batch_resumes_1'

In [49]:
# Generate DataFrame (treating each page in the batch as 1 candidate)
df_resumes = parse_batch_pdfs_regex(batch_dir)

# Inspect top rows
print(df_resumes.head())

candidate_count: 0
page_email: rahulmalik@email.com
page_phone: 1234567890
candidate_count: 1
page_email: j.hughes@email.com
page_phone: 1234567890
candidate_count: 2
page_email: p.fox@email.com
page_phone: 1234567890
candidate_count: 3
page_email: tcoleman@email.com
page_phone: 1234567890
candidate_count: 4
page_email: l.simmons@email.com
page_phone: 1234567890
candidate_count: 0
page_email: r.patel@email.com
page_phone: 1234567890
candidate_count: 1
page_email: farahmartin@email.com
page_phone: 1234567890
candidate_count: 2
page_email: a.thorne@email.com
page_phone: 1234567890
candidate_count: 3
page_email: e.santos@email.com
page_phone: 1234567890
candidate_count: 4
page_email: d.garcia@email.com
page_phone: 1234567890
candidate_count: 0
page_email: alonsoromero@email.com
page_phone: 1234567890
candidate_count: 1
page_email: m.hernandez@email.com
page_phone: 1234567890
candidate_count: 2
page_email: tomislav.abram@email.com
page_phone: 1234567890
Processed 3 PDF batches into 13 cand

In [28]:
df_resumes.head()

,file_name,candidate_id,parsed_resume_text
0,batch_resumes_1.pdf,batch_resumes_1_p1,RAHUL MALIK\nNLP DATA SCIENTIST\nCONTACT\nrahu...
1,batch_resumes_1.pdf,batch_resumes_1_p2,JACK HUGHES\nSenior Data Scientist\nj.hughes@e...
2,batch_resumes_1.pdf,batch_resumes_1_p3,CONTACT\np.fox@email.com\n(123) 456-7890\nLaur...
3,batch_resumes_1.pdf,batch_resumes_1_p4,TERRENCE\nCOLEMAN\nSenior Data Scientist\ntcol...
4,batch_resumes_1.pdf,batch_resumes_1_p5,LUCAS SIMMONS\nSENIOR DATA SCIENTIST\nCONTACT\...


In [45]:
df_resumes.shape

(13, 5)

In [47]:
df_resumes.candidate_id.tolist()

['batch_resumes_1_cand_1',
 'batch_resumes_1_cand_2',
 'batch_resumes_1_cand_3',
 'batch_resumes_1_cand_4',
 'batch_resumes_1_cand_5',
 'batch_resumes_2_cand_1',
 'batch_resumes_2_cand_2',
 'batch_resumes_2_cand_3',
 'batch_resumes_2_cand_4',
 'batch_resumes_2_cand_5',
 'batch_resumes_3_cand_1',
 'batch_resumes_3_cand_2',
 'batch_resumes_3_cand_3']